# Module 06 — Full Agent: FoodieBuddy + Guardrails

In module 05 you learned how AgentCore memory works in isolation. Here we wire everything together into a production-ready agent and add one more layer: **Amazon Bedrock Guardrails**.

### What we're adding

| Feature | Where it comes from |
|---|---|
| Agent + model | Module 01 |
| `@tool` for food search | Module 02 |
| System prompt with persona | Module 03 |
| Memory hooks (init + after invocation) | Module 04 + 05 |
| Persistent memory (AgentCore) | Module 05 |
| **Guardrails** | This module |

---

### What are Bedrock Guardrails?

Guardrails let you define safety policies that are enforced at the API level — independent of the model and your prompt. They can:

- **Block topics** — e.g., refuse to discuss anything unrelated to food
- **Filter harmful content** — hate speech, violence, sexual content (configurable thresholds)
- **Redact PII** — automatically mask emails, phone numbers, names in responses
- **Block specific words** — custom deny-lists
- **Ground responses** — detect hallucinations against a provided context (RAG use cases)

Guardrails apply to both the input (user message) and the output (model response).

```
User message → [Guardrail INPUT check] → Model → [Guardrail OUTPUT check] → Response
                      ↓ blocked?                         ↓ blocked?
                 Return denial msg                  Return denial msg
```

In [ ]:
import os
import logging
import boto3
from datetime import datetime

from strands import Agent, tool
from strands.hooks import (
    AgentInitializedEvent,
    AfterInvocationEvent,
    HookProvider,
    HookRegistry
)
from bedrock_agentcore.memory import MemoryClient
from ddgs import DDGS
from ddgs.exceptions import RatelimitException

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger("foodie-buddy")

REGION = os.getenv('AWS_REGION', 'us-east-1')
USER_ID = "food-lover-001"
SESSION_ID = f"food_chat_{datetime.now().strftime('%Y%m%d%H%M%S')}"

# Paste the memory_id you got from 05_memory.ipynb
MEMORY_ID = "FoodAgentMemory-Dkm2Ve6eDw"  # <-- replace this

print(f"Region     : {REGION}")
print(f"User ID    : {USER_ID}")
print(f"Session ID : {SESSION_ID}")
print(f"Memory ID  : {MEMORY_ID}")

### Step 1 — Create a Bedrock Guardrail

We create a guardrail that:
- Blocks off-topic conversations (anything not food-related)
- Filters harmful content
- Redacts any PII that slips into responses

You only need to create this once — save the `guardrail_id` and `guardrail_version` for reuse.

In [ ]:
bedrock = boto3.client('bedrock', region_name=REGION)

try:
    response = bedrock.create_guardrail(
        name="FoodieBuddyGuardrail",
        description="Keeps FoodieBuddy focused on food topics and safe responses",

        # Block off-topic subjects
        topicPolicyConfig={
            "topicsConfig": [
                {
                    "name": "Non-food topics",
                    "definition": "Any conversation not related to food, cooking, recipes, restaurants, nutrition, or dietary advice",
                    "examples": [
                        "Tell me about the stock market",
                        "Help me write code",
                        "What is the weather today"
                    ],
                    "type": "DENY"
                }
            ]
        },

        # Filter harmful content
        contentPolicyConfig={
            "filtersConfig": [
                {"type": "HATE",     "inputStrength": "HIGH", "outputStrength": "HIGH"},
                {"type": "VIOLENCE", "inputStrength": "MEDIUM", "outputStrength": "MEDIUM"},
                {"type": "SEXUAL",   "inputStrength": "HIGH", "outputStrength": "HIGH"},
            ]
        },

        # Redact PII from responses
        sensitiveInformationPolicyConfig={
            "piiEntitiesConfig": [
                {"type": "EMAIL",        "action": "ANONYMIZE"},
                {"type": "PHONE",        "action": "ANONYMIZE"},
                {"type": "NAME",         "action": "ANONYMIZE"},
            ]
        },

        blockedInputMessaging="I can only help with food-related topics. Ask me about recipes, cuisines, or restaurants!",
        blockedOutputsMessaging="I can only provide food-related information. Let me know if you have a food question!"
    )

    guardrail_id = response['guardrailId']
    print(f"Created guardrail: {guardrail_id}")

    # Publish a version so it can be used
    version_response = bedrock.create_guardrail_version(guardrailIdentifier=guardrail_id)
    guardrail_version = version_response['version']
    print(f"Guardrail version : {guardrail_version}")
    print("\nSave these for reuse:")
    print(f"  GUARDRAIL_ID      = '{guardrail_id}'")
    print(f"  GUARDRAIL_VERSION = '{guardrail_version}'")

except bedrock.exceptions.ConflictException:
    # Already exists — list and reuse
    guardrails = bedrock.list_guardrails()['guardrails']
    existing = next((g for g in guardrails if g['name'] == 'FoodieBuddyGuardrail'), None)
    guardrail_id = existing['id']
    guardrail_version = existing['version']
    print(f"Reusing existing guardrail: {guardrail_id} v{guardrail_version}")

### Step 2 — Test the guardrail directly

Before wiring it into the agent, let's verify it works as expected.

In [ ]:
bedrock_runtime = boto3.client('bedrock-runtime', region_name=REGION)

def test_guardrail(text: str, source: str = "INPUT"):
    """Test a guardrail against a piece of text."""
    response = bedrock_runtime.apply_guardrail(
        guardrailIdentifier=guardrail_id,
        guardrailVersion=guardrail_version,
        source=source,
        content=[{"text": {"text": text}}]
    )
    action = response['action']  # NONE or GUARDRAIL_INTERVENED
    print(f"Input  : {text!r}")
    print(f"Action : {action}")
    if action == 'GUARDRAIL_INTERVENED':
        for assessment in response.get('assessments', []):
            topic = assessment.get('topicPolicy', {})
            for t in topic.get('topics', []):
                if t['action'] == 'BLOCKED':
                    print(f"Blocked by topic: {t['name']}")
    print()

# Should pass — food topic
test_guardrail("What are some good vegetarian South Indian recipes?")

# Should be blocked — off-topic
test_guardrail("What is the current Bitcoin price?")

### Step 3 — Memory hook (reused from module 05)

Same hook as before — loads preferences on init, saves turns after each invocation.

In [ ]:
memory_client = MemoryClient(region_name=REGION)

class FoodMemoryHookProvider(HookProvider):
    """Manages persistent memory for the food agent."""

    def __init__(self, memory_client: MemoryClient, memory_id: str):
        self.memory_client = memory_client
        self.memory_id = memory_id

    def on_agent_initialized(self, event: AgentInitializedEvent):
        try:
            actor_id = event.agent.state.get("actor_id")
            if not actor_id:
                return
            preferences = self.memory_client.retrieve_memories(
                memory_id=self.memory_id,
                namespace=f"user/{actor_id}/food_preferences",
                query="food preferences cuisines dietary restrictions favorites allergies",
                top_k=5
            )
            if preferences:
                lines = []
                for pref in preferences:
                    text = pref.get('content', {}).get('text', '').strip()
                    if text:
                        lines.append(f"- {text}")
                if lines:
                    event.agent.system_prompt += f"\n\n## User's Food Preferences:\n" + "\n".join(lines)
                    logger.info(f"Loaded {len(lines)} preferences from memory")
            else:
                logger.info("No previous preferences found")
        except Exception as e:
            logger.error(f"Error loading preferences: {e}")

    def on_after_invocation(self, event: AfterInvocationEvent):
        try:
            messages = event.agent.messages
            if len(messages) < 2:
                return
            actor_id = event.agent.state.get("actor_id")
            session_id = event.agent.state.get("session_id")
            if not actor_id or not session_id:
                return
            user_msg = assistant_msg = None
            for msg in reversed(messages):
                if msg["role"] == "assistant" and not assistant_msg:
                    c = msg.get("content", [])
                    if c and "text" in c[0]:
                        assistant_msg = c[0]["text"]
                elif msg["role"] == "user" and not user_msg:
                    c = msg.get("content", [])
                    if c and "text" in c[0] and "toolResult" not in c[0]:
                        user_msg = c[0]["text"]
                        break
            if user_msg and assistant_msg:
                self.memory_client.create_event(
                    memory_id=self.memory_id,
                    actor_id=actor_id,
                    session_id=session_id,
                    messages=[(user_msg, "USER"), (assistant_msg, "ASSISTANT")]
                )
                logger.info("Saved conversation turn to memory")
        except Exception as e:
            logger.error(f"Error saving conversation: {e}")

    def register_hooks(self, registry: HookRegistry):
        registry.add_callback(AgentInitializedEvent, self.on_agent_initialized)
        registry.add_callback(AfterInvocationEvent, self.on_after_invocation)

### Step 4 — Food search tool

In [ ]:
@tool
def search_food(query: str, max_results: int = 5) -> str:
    """Search for food information, recipes, cuisines, or restaurant recommendations.

    Args:
        query: Search query about food
        max_results: Maximum number of results to return

    Returns:
        Search results with food information
    """
    try:
        results = DDGS().text(f"{query} food recipe restaurant", region="us-en", max_results=max_results)
        if not results:
            return "No results found."
        return "\n\n".join(
            f"{i}. {r.get('title', '')}\n   {r.get('body', '')}"
            for i, r in enumerate(results, 1)
        )
    except RatelimitException:
        return "Rate limit reached. Please try again in a moment."
    except Exception as e:
        return f"Search error: {e}"

### Step 5 — Create the agent with guardrails

Pass `guardrail_config` to the `Agent`. Strands forwards it to every Bedrock API call automatically.

In [ ]:
def create_food_agent(user_id: str, session_id: str) -> Agent:
    from strands.models import BedrockModel

    system_prompt = f"""You are FoodieBuddy, a friendly food assistant.
Help users discover new foods and remember their preferences.
Today's date: {datetime.today().strftime('%Y-%m-%d')}
Always respect dietary restrictions and allergies.
Keep responses concise and practical.
"""
    model = BedrockModel(
        model_id="us.anthropic.claude-3-5-haiku-20241022-v1:0",
        guardrail_id=guardrail_id,
        guardrail_version=guardrail_version,
        guardrail_trace="enabled"
    )
    return Agent(
        name="FoodieBuddy",
        model=model,
        system_prompt=system_prompt,
        hooks=[FoodMemoryHookProvider(memory_client, MEMORY_ID)],
        tools=[search_food],
        state={"actor_id": user_id, "session_id": session_id}
    )


food_agent = create_food_agent(USER_ID, SESSION_ID)
logger.info("FoodieBuddy with guardrails ready!")

### Step 6 — Chat and see guardrails in action

In [ ]:
# Normal food question — should work fine
print("You: I love spicy food, especially South Indian cuisines! But I'm vegetarian.")
print("\nFoodieBuddy: ", end="")
food_agent("I love spicy food, especially South Indian cuisines! But I'm vegetarian.")

In [ ]:
# Off-topic question — guardrail should block this
print("You: Can you help me write a Python script to scrape websites?")
print("\nFoodieBuddy: ", end="")
food_agent("Can you help me write a Python script to scrape websites?")

In [ ]:
# Back to food — should work, and memory from previous turn is still active
print("You: What should I make for dinner tonight?")
print("\nFoodieBuddy: ", end="")
food_agent("What should I make for dinner tonight?")

### Step 7 — New session: memory + guardrails together

Start a fresh session. The agent loads preferences from module 05's memory and still has guardrails active.

In [ ]:
NEW_SESSION_ID = f"food_chat_{datetime.now().strftime('%Y%m%d%H%M%S')}_new"
food_agent_2 = create_food_agent(USER_ID, NEW_SESSION_ID)

print("You: Hi! Do you remember what I like? What do you recommend for tonight?")
print("\nFoodieBuddy: ", end="")
food_agent_2("Hi! Do you remember what I like? What do you recommend for tonight?")

---

### Key takeaways

- `bedrock.create_guardrail()` defines your safety policy — topics, content filters, PII redaction
- `bedrock.create_guardrail_version()` publishes it for use
- Pass `guardrail_config` to `Agent(...)` — Strands applies it to every model call automatically
- Guardrails are enforced at the API level, independent of your prompt
- `trace: enabled` lets you see exactly which policy triggered an intervention

### What's next?

- Deploy FoodieBuddy as a REST API using `bedrock-agentcore`
- Build a multi-agent system where FoodieBuddy delegates to specialist agents (nutrition, recipes, restaurants)
- Add grounding checks to detect hallucinations in recipe suggestions
- Integrate with Kiro hooks to automate agent workflows in your IDE